# Part Lookup Sheet Generator

Builds a **"Part Lookup"** sheet on top of an FD Processed workbook (same template as
`FD_Processed_Cummins_Steron_BIN_*.xlsx`) — a dropdown-driven, single-part view of:

- Part summary (RC, Total Calls, Unit Price, FD Final, Max, On Hand, On Order, Trend Coef, %Accum., Alert)
- Monthly detail — units (Incoming / Est. OH / Est. OO / Sched. Order)
- Monthly detail — dollar amounts (Amt. Est. OO / Amt. Sched. Order)
- A trend chart for the selected part

The sheet is driven entirely by `INDEX`/`MATCH` formulas against column **headers**, not fixed
column letters — so it keeps working if the source file gains/loses rows, or if columns shift
position, as long as the header text itself (`"Est. OH M-1"`, `"Sched. Order M-3"`, etc.) stays
the same as this template.

**Note on formulas:** openpyxl writes formulas but not their cached results. Open the output
file in Excel and it will recalculate automatically — no extra step needed. If you want to
verify values *before* opening Excel (e.g. in an automated pipeline), see the optional recalc
cell at the bottom.

In [1]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.chart import LineChart, Reference
from openpyxl.utils import get_column_letter

In [2]:
# ============================================================
# CONFIG — change these per run
# ============================================================
INPUT_FILE = "FD_Processed_Cummins_Steron_BIN_29Jul26.xlsx"   # input workbook (same template each time)
OUTPUT_FILE = "FD_Processed_with_PN_Lookup.xlsx"               # where to save the result

SOURCE_SHEET_NAME = "FD Processed"   # tab holding the raw FD Processed data
LOOKUP_SHEET_NAME = "Part Lookup"    # tab this script creates/replaces

PN_COLUMN_HEADER = "P/N"             # header of the part-number column in the source sheet
DESC_COLUMN_HEADER = "Desc"

# Fields shown in the "Part Summary" strip — (display label, exact source header text)
SUMMARY_FIELDS = [
    ("RC", "RC"),
    ("Total Calls", "Total Calls"),
    ("Unit Price", "DN Price"),
    ("FD Final", "FD_final"),
    ("Max", "Max"),
    ("On Hand", "OH"),
    ("On Order", "OO"),
    ("Trend Coef", "Trend Coef"),
    ("%Accum.", "%Accum."),
    ("Alert Score", "forecast_alert_score"),
    ("Alert", "forecast_alert_label"),
]

# Monthly metrics grid — row label must be the exact prefix used in the source headers
# (e.g. source column "Est. OH M-3" = "Est. OH" + " " + "M-3")
UNIT_METRICS = ["Incoming", "Est. OH", "Est. OO", "Sched. Order"]
AMOUNT_METRICS = ["Amt. Est. OO", "Amt. Sched. Order"]
MONTH_LABELS = [f"M-{i}" for i in range(1, 10)]   # M-1 .. M-9 — covers the widest metric (Est. OH)


In [3]:
wb = openpyxl.load_workbook(INPUT_FILE, data_only=False)
src = wb[SOURCE_SHEET_NAME]

last_row = src.max_row               # last row of data (header + parts)
last_col_letter = get_column_letter(src.max_column)   # last column, e.g. "BO"
default_pn = str(src.cell(row=2, column=1).value)     # first P/N in the sheet, used as the default dropdown value

print(f"Source sheet: {SOURCE_SHEET_NAME!r}  |  rows: {last_row}  |  columns: A:{last_col_letter}  |  default P/N: {default_pn}")

# Drop any previous version of the lookup sheet so re-running this notebook is safe
if LOOKUP_SHEET_NAME in wb.sheetnames:
    del wb[LOOKUP_SHEET_NAME]
ws = wb.create_sheet(LOOKUP_SHEET_NAME, 0)   # inserted as the first tab


Source sheet: 'FD Processed'  |  rows: 2183  |  columns: A:BO  |  default P/N: 3052231


In [4]:
# ============================================================
# Styles
# ============================================================
NAVY = "1F2A3A"
LIGHT_GREY = "F4F6F8"
MONTH_BLUE = "D9E2EC"

white_font = Font(color="FFFFFF", bold=True, size=13, name="Arial")
title_fill = PatternFill("solid", fgColor=NAVY)

label_font = Font(bold=True, name="Arial", size=10, color="33414F")
value_font = Font(name="Arial", size=10, color="000000")

section_font = Font(bold=True, name="Arial", size=10, color="FFFFFF")
section_fill = PatternFill("solid", fgColor="33414F")

lightgrey_fill = PatternFill("solid", fgColor=LIGHT_GREY)
month_fill = PatternFill("solid", fgColor=MONTH_BLUE)

thin = Side(style="thin", color="C9D2DA")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

input_fill = PatternFill("solid", fgColor="FFF6E5")
input_font = Font(bold=True, name="Arial", size=12, color="000000")


def set_cell(ws, cell_ref, value, font=None, fill=None, align=None, numfmt=None, brd=None):
    """Write a value/formula into a cell with optional styling."""
    c = ws[cell_ref]
    c.value = value
    if font: c.font = font
    if fill: c.fill = fill
    if align: c.alignment = align
    if numfmt: c.number_format = numfmt
    if brd: c.border = brd
    return c


def two_way_lookup(row_key_ref, header_text_expr, source_sheet=SOURCE_SHEET_NAME,
                    last_col_letter=None):
    """
    Build an =IFERROR(INDEX(...), "") formula that looks up `row_key_ref` (a P/N, e.g. "$B$4")
    down the source sheet's first column, and `header_text_expr` (a formula fragment that
    evaluates to an exact source header string, e.g. '"RC"' or '$A12&" "&B$11')
    across the source sheet's header row.
    """
    return (
        f'=IFERROR(INDEX(\'{source_sheet}\'!$A:${last_col_letter}, '
        f'MATCH({row_key_ref},\'{source_sheet}\'!$A:$A,0), '
        f'MATCH({header_text_expr},\'{source_sheet}\'!$A$1:${last_col_letter}$1,0)), "")'
    )


In [5]:
# ============================================================
# Title + P/N selector
# ============================================================
ws.merge_cells("A1:K2")
set_cell(ws, "A1", "PART NUMBER LOOKUP", white_font, title_fill,
          Alignment(vertical="center", horizontal="left", indent=1))
for row in ws["A1:K2"]:
    for c in row:
        c.fill = title_fill

set_cell(ws, "A4", "Select P/N:", label_font)
set_cell(ws, "B4", default_pn, input_font, input_fill, Alignment(horizontal="center"), brd=border)
ws["B4"].number_format = "@"   # keep as text — the source P/N column is text, not numeric

set_cell(ws, "C4", "Description:", label_font)
ws.merge_cells("D4:K4")
desc_formula = two_way_lookup("$B$4", f'"{DESC_COLUMN_HEADER}"', last_col_letter=last_col_letter)
desc_formula = desc_formula.replace('"")', '"Part not found")')  # friendlier fallback text
set_cell(ws, "D4", desc_formula, Font(italic=True, name="Arial", size=10))

# Dropdown of every P/N in the source sheet
dv = DataValidation(type="list", formula1=f"='{SOURCE_SHEET_NAME}'!$A$2:$A${last_row}",
                     allow_blank=False, showDropDown=False)
dv.error = f"Please select a valid P/N from the {SOURCE_SHEET_NAME} sheet."
dv.errorTitle = "Invalid P/N"
ws.add_data_validation(dv)
dv.add(ws["B4"])


In [6]:
# ============================================================
# Part Summary section
# ============================================================
ws.merge_cells("A6:K6")
set_cell(ws, "A6", "PART SUMMARY", section_font, section_fill)

for i, (label, header) in enumerate(SUMMARY_FIELDS):
    col = get_column_letter(1 + i)
    set_cell(ws, f"{col}7", label, label_font, lightgrey_fill, Alignment(horizontal="center"), brd=border)
    formula = two_way_lookup("$B$4", f'"{header}"', last_col_letter=last_col_letter)
    numfmt = None
    if header == "DN Price": numfmt = "$#,##0.00"
    elif header == "%Accum.": numfmt = '0.0"%"'
    elif header == "Trend Coef": numfmt = "0.00"
    set_cell(ws, f"{col}8", formula, value_font, None, Alignment(horizontal="center"), numfmt=numfmt, brd=border)


In [7]:
# ============================================================
# Monthly Detail — Units
# ============================================================
ws.merge_cells("A10:J10")
set_cell(ws, "A10", "MONTHLY DETAIL — UNITS", section_font, section_fill)

set_cell(ws, "A11", "Metric", label_font, lightgrey_fill, Alignment(horizontal="center"), brd=border)
for i, m in enumerate(MONTH_LABELS):
    col = get_column_letter(2 + i)
    set_cell(ws, f"{col}11", m, label_font, month_fill, Alignment(horizontal="center"), brd=border)

row = 12
for metric in UNIT_METRICS:
    set_cell(ws, f"A{row}", metric, label_font, lightgrey_fill, Alignment(horizontal="left", indent=1), brd=border)
    for i, m in enumerate(MONTH_LABELS):
        col = get_column_letter(2 + i)
        header_expr = f'$A{row}&" "&{col}$11'   # e.g. "Est. OH"&" "&"M-3" -> "Est. OH M-3"
        formula = two_way_lookup("$B$4", header_expr, last_col_letter=last_col_letter)
        set_cell(ws, f"{col}{row}", formula, value_font, None, Alignment(horizontal="center"),
                  numfmt="#,##0", brd=border)
    row += 1

units_last_row = row - 1   # last metric row, needed later for the chart


In [8]:
# ============================================================
# Monthly Detail — Dollar Amounts
# ============================================================
amt_header_row = units_last_row + 2
ws.merge_cells(f"A{amt_header_row}:J{amt_header_row}")
set_cell(ws, f"A{amt_header_row}", "MONTHLY DETAIL — DOLLAR AMOUNTS", section_font, section_fill)

col_header_row = amt_header_row + 1
set_cell(ws, f"A{col_header_row}", "Metric", label_font, lightgrey_fill, Alignment(horizontal="center"), brd=border)
for i, m in enumerate(MONTH_LABELS):
    col = get_column_letter(2 + i)
    set_cell(ws, f"{col}{col_header_row}", m, label_font, month_fill, Alignment(horizontal="center"), brd=border)

row = col_header_row + 1
for metric in AMOUNT_METRICS:
    set_cell(ws, f"A{row}", metric, label_font, lightgrey_fill, Alignment(horizontal="left", indent=1), brd=border)
    for i, m in enumerate(MONTH_LABELS):
        col = get_column_letter(2 + i)
        header_expr = f'$A{row}&" "&{col}${col_header_row}'
        formula = two_way_lookup("$B$4", header_expr, last_col_letter=last_col_letter)
        set_cell(ws, f"{col}{row}", formula, value_font, None, Alignment(horizontal="center"),
                  numfmt="$#,##0.00", brd=border)
    row += 1

amounts_last_row = row - 1


In [9]:
# ============================================================
# Trend chart — units across months for the selected part
# ============================================================
chart_anchor_row = amounts_last_row + 2

chart = LineChart()
chart.title = "Monthly Units Trend — Incoming / Est. OH / Est. OO / Sched. Order"
chart.style = 2
chart.y_axis.title = "Units"
chart.x_axis.title = "Month"
chart.width = 24
chart.height = 10

cats = Reference(ws, min_col=2, max_col=1 + len(MONTH_LABELS), min_row=11, max_row=11)
data = Reference(ws, min_col=1, max_col=1 + len(MONTH_LABELS), min_row=12, max_row=units_last_row)
chart.add_data(data, titles_from_data=True, from_rows=True)
chart.set_categories(cats)

ws.add_chart(chart, f"A{chart_anchor_row}")

# Column widths
ws.column_dimensions["A"].width = 20
for col in "BCDEFGHIJK":
    ws.column_dimensions[col].width = 13


In [10]:
wb.save(OUTPUT_FILE)
print(f"Saved: {OUTPUT_FILE}")


Saved: FD_Processed_with_PN_Lookup.xlsx


## Optional — verify formula values before opening in Excel

openpyxl never computes formula results, so any tool reading the file with
`data_only=True` (pandas, a quick preview, etc.) will see blank cells until the workbook has
been opened and saved once in real Excel, or recalculated headlessly with LibreOffice below.
**This step is optional** — opening `OUTPUT_FILE` directly in Excel recalculates everything
automatically.

In [11]:
# Requires LibreOffice ('soffice') installed locally — skip this cell if you don't have it.
# import subprocess
# result = subprocess.run(["soffice", "--headless", "--convert-to", "xlsx",
#                           "--outdir", ".", OUTPUT_FILE], capture_output=True, text=True)
# print(result.stdout, result.stderr)
